# Qwen Teacher T4 Notebook

Stable Kaggle single-T4 teacher generation flow.

Run order:
1. Setup
2. Config
3. Input validation
4. Model load
5. Smoke run
6. Smoke inspection
7. Full run
8. Final inspection

Notes:
- Use a single `NvidiaTeslaT4` runtime.
- Leave `INPUT_PATH_OVERRIDE` empty unless you intentionally want a non-default mounted path.
- Review the smoke run before enabling the full run.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import sys

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

EMBEDDED_RUNTIME_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport json\nimport os\nimport re\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\n\nDEFAULT_MODEL = "Qwen/Qwen3-4B-Instruct-2507"\nDEFAULT_TRANSCRIPT_TURN_CHAR_LIMIT = 1200\nDEFAULT_TRANSCRIPT_CHAR_BUDGET = 14000\nDEFAULT_DEBUG_TEXT_CHAR_LIMIT = 6000\n\nPREAMBLE = "We are continuing an earlier conversation, not starting fresh."\nTHINK_BLOCK_PATTERN = re.compile(r"<think>([\\s\\S]*?)</think>", re.IGNORECASE)\nREQUIRED_SECTIONS = [\n    PREAMBLE,\n    "Objective:",\n    "Established context:",\n    "Decisions already made:",\n    "Rejected paths / do not revisit unless necessary:",\n    "Open questions:",\n    "Where we left off:",\n    "Continue from this exact state.",\n    "My next request:",\n]\nSECTION_LINE_PATTERNS = {\n    PREAMBLE: re.compile(rf"^{re.escape(PREAMBLE)}$", re.MULTILINE),\n    "Objective:": re.compile(r"^Objective:$", re.MULTILINE),\n    "Established context:": re.compile(r"^Established context:$", re.MULTILINE),\n    "Decisions already made:": re.compile(r"^Decisions already made:$", re.MULTILINE),\n    "Rejected paths / do not revisit unless necessary:": re.compile(\n        r"^Rejected paths / do not revisit unless necessary:$",\n        re.MULTILINE,\n    ),\n    "Open questions:": re.compile(r"^Open questions:$", re.MULTILINE),\n    "Where we left off:": re.compile(r"^Where we left off:$", re.MULTILINE),\n    "Continue from this exact state.": re.compile(r"^Continue from this exact state\\.$", re.MULTILINE),\n    "My next request:": re.compile(r"^My next request:$", re.MULTILINE),\n}\nPROMPT_ECHO_MARKERS = (\n    "\\nRequirements:\\n",\n    "\\nKey requirements:\\n",\n    "Exactly the structure and section order:",\n    "Rewrite the supplied conversation state into the exact brief template below.",\n    "Return exactly this structure:",\n    "\\nSource transcript:\\n",\n    "Do not invent goals, decisions, constraints, files, APIs, or next steps.",\n    "We are given a transcript",\n    "We have the rolling state helper provided",\n    "We have the transcript from the conversation",\n    "Let\'s break down",\n)\nPLACEHOLDER_BULLET_PATTERN = re.compile(r"(?m)^-\\s*\\.\\.\\.\\s*$")\nINLINE_SECTION_PATTERNS = {\n    "Objective:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Objective\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n    "Established context:": re.compile(\n        r"(?mi)^\\s*(?:\\*\\*)?\\s*Established context\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"\n    ),\n    "Decisions already made:": re.compile(\n        r"(?mi)^\\s*(?:\\*\\*)?\\s*Decisions already made\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"\n    ),\n    "Rejected paths / do not revisit unless necessary:": re.compile(\n        r"(?mi)^\\s*(?:\\*\\*)?\\s*Rejected paths / do not revisit unless necessary\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"\n    ),\n    "Open questions:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Open questions\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n    "Where we left off:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Where we left off\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n    "My next request:": re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*My next request\\s*(?:\\*\\*)?\\s*:[ \\t]+(.+\\S)\\s*$"),\n}\nSECTION_HEADER_NORMALIZATIONS = (\n    (re.compile(rf"(?mi)^\\s*{re.escape(PREAMBLE)}\\s*$"), PREAMBLE),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Objective\\s*(?:\\*\\*)?\\s*:\\s*$"), "Objective:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Established context\\s*(?:\\*\\*)?\\s*:\\s*$"), "Established context:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Decisions already made\\s*(?:\\*\\*)?\\s*:\\s*$"), "Decisions already made:"),\n    (\n        re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Rejected paths / do not revisit unless necessary\\s*(?:\\*\\*)?\\s*:\\s*$"),\n        "Rejected paths / do not revisit unless necessary:",\n    ),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Open questions\\s*(?:\\*\\*)?\\s*:\\s*$"), "Open questions:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Where we left off\\s*(?:\\*\\*)?\\s*:\\s*$"), "Where we left off:"),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*Continue from this exact state\\.?\\s*$"), "Continue from this exact state."),\n    (re.compile(r"(?mi)^\\s*(?:\\*\\*)?\\s*My next request\\s*(?:\\*\\*)?\\s*:\\s*$"), "My next request:"),\n)\nKNOWN_INPUT_PATHS = {\n    "full": "/kaggle/input/qwen4b-teacher-2gpu-inputs/teacher_input_rows.json",\n    "repair": "/kaggle/input/qwen4b-teacher-2gpu-repair-inputs/live_extension_review_queue_parallel_assistant_repaired_v11.json",\n}\n\n\nclass MalformedBriefError(ValueError):\n    def __init__(self, issues: list[str], debug_payload: dict[str, Any]) -> None:\n        self.issues = issues\n        self.debug_payload = debug_payload\n        super().__init__(f"Model returned a malformed continuation brief: {summarize_issues(issues)}")\n\n\ndef ensure_parent(path: str | None) -> None:\n    if path:\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n\n\ndef sanitize_text(text: Any) -> str:\n    return re.sub(r"\\s+", " ", str(text or "")).strip()\n\n\ndef trim_text(text: Any, max_chars: int) -> str:\n    compact = sanitize_text(text)\n    if len(compact) <= max_chars:\n        return compact\n    return f"{compact[:max_chars].strip()}..."\n\n\ndef trim_debug_text(text: Any, max_chars: int = DEFAULT_DEBUG_TEXT_CHAR_LIMIT) -> str:\n    value = str(text or "")\n    if len(value) <= max_chars:\n        return value\n    remaining = len(value) - max_chars\n    return f"{value[:max_chars].rstrip()}\\n...[truncated {remaining} chars]"\n\n\ndef build_transcript_text(\n    transcript_turns: list[dict[str, Any]],\n    *,\n    per_turn_char_limit: int,\n    total_char_budget: int,\n) -> str:\n    rendered_turns = [\n        f"{index + 1}. {str(turn.get(\'role\', \'\')).upper()}: {trim_text(turn.get(\'text\', \'\'), per_turn_char_limit)}"\n        for index, turn in enumerate(transcript_turns)\n    ]\n    if not rendered_turns:\n        return "No transcript turns were provided."\n    if total_char_budget <= 0:\n        return "\\n".join(rendered_turns)\n\n    kept_turns: list[str] = []\n    used_chars = 0\n    for turn_text in reversed(rendered_turns):\n        turn_size = len(turn_text) + 1\n        if kept_turns and used_chars + turn_size > total_char_budget:\n            break\n        kept_turns.append(turn_text)\n        used_chars += turn_size\n        if used_chars >= total_char_budget:\n            break\n\n    kept_turns.reverse()\n    omitted_turns = len(rendered_turns) - len(kept_turns)\n    if omitted_turns > 0:\n        kept_turns.insert(0, f"[{omitted_turns} earlier turns omitted to fit the transcript budget.]")\n    return "\\n".join(kept_turns)\n\n\ndef render_list(title: str, items: list[str], fallback: str) -> str:\n    lines = items if items else [fallback]\n    return f"{title}:\\n" + "\\n".join(f"- {item}" for item in lines)\n\n\ndef render_template_list(items: list[str], fallback: str) -> str:\n    lines = [item for item in items if item]\n    return "\\n".join(f"- {item}" for item in (lines or [fallback]))\n\n\ndef build_prompt(\n    row: dict[str, Any],\n    capture_rationale: bool,\n    *,\n    transcript_char_budget: int = DEFAULT_TRANSCRIPT_CHAR_BUDGET,\n    per_turn_char_limit: int = DEFAULT_TRANSCRIPT_TURN_CHAR_LIMIT,\n) -> str:\n    rolling_state = row.get("rolling_state") or {}\n    transcript_turns = row.get("transcript_turns") or []\n    transcript_text = build_transcript_text(\n        transcript_turns,\n        per_turn_char_limit=per_turn_char_limit,\n        total_char_budget=transcript_char_budget,\n    )\n    objective = render_template_list(\n        [sanitize_text(rolling_state.get("objective"))] if sanitize_text(rolling_state.get("objective")) else [],\n        "Objective still needs confirmation.",\n    )\n    context_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("constraints") or []) if sanitize_text(item)],\n        "No explicit constraints have been captured yet.",\n    )\n    decision_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("decisions") or []) if sanitize_text(item)],\n        "No firm decisions are recorded yet.",\n    )\n    rejected_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("rejected") or []) if sanitize_text(item)],\n        "No rejected paths have been recorded yet.",\n    )\n    open_question_lines = render_template_list(\n        [sanitize_text(item) for item in (rolling_state.get("open_questions") or []) if sanitize_text(item)],\n        "No open questions are active right now.",\n    )\n    next_step_lines = render_template_list(\n        [sanitize_text(rolling_state.get("next_step"))] if sanitize_text(rolling_state.get("next_step")) else [],\n        "Resume from the latest visible turn.",\n    )\n\n    instructions = [\n        "Rewrite the supplied conversation state into the exact brief template below.",\n        "Keep every section header exactly as written.",\n        "Do not add any extra headers, titles, dates, explanations, or code fences.",\n        "Use the transcript as the source of truth and the rolling state only as a helper.",\n        "If something is unclear or not explicit in the transcript, leave it out.",\n        "If the supplied content is already concise and faithful, copy it with only minimal edits.",\n        "Return only the final brief text.",\n    ]\n    if capture_rationale:\n        instructions.extend(\n            [\n                "If you show visible reasoning, place it inside <think>...</think>.",\n                "After the closing </think> tag, return only the final continuation brief.",\n            ]\n        )\n\n    return "\\n".join(\n        [\n            *instructions,\n            "",\n            "Source transcript:",\n            f"- title: {row.get(\'title\', \'\')}",\n            f"- provider: {row.get(\'provider\', \'\')}",\n            transcript_text or "No transcript turns were provided.",\n            "",\n            "Rolling state helper:",\n            render_list(\n                "Objective",\n                [sanitize_text(rolling_state.get("objective"))] if sanitize_text(rolling_state.get("objective")) else [],\n                "Objective still needs confirmation.",\n            ),\n            "",\n            render_list(\n                "Established context",\n                [sanitize_text(item) for item in (rolling_state.get("constraints") or []) if sanitize_text(item)],\n                "No explicit constraints have been captured yet.",\n            ),\n            "",\n            render_list(\n                "Decisions already made",\n                [sanitize_text(item) for item in (rolling_state.get("decisions") or []) if sanitize_text(item)],\n                "No firm decisions are recorded yet.",\n            ),\n            "",\n            render_list(\n                "Rejected paths / do not revisit unless necessary",\n                [sanitize_text(item) for item in (rolling_state.get("rejected") or []) if sanitize_text(item)],\n                "No rejected paths have been recorded yet.",\n            ),\n            "",\n            render_list(\n                "Open questions",\n                [sanitize_text(item) for item in (rolling_state.get("open_questions") or []) if sanitize_text(item)],\n                "No open questions are active right now.",\n            ),\n            "",\n            render_list(\n                "Where we left off",\n                [sanitize_text(rolling_state.get("next_step"))] if sanitize_text(rolling_state.get("next_step")) else [],\n                "Resume from the latest visible turn.",\n            ),\n            "",\n            "Return exactly this structure:",\n            "",\n            PREAMBLE,\n            "",\n            "Objective:",\n            objective,\n            "",\n            "Established context:",\n            context_lines,\n            "",\n            "Decisions already made:",\n            decision_lines,\n            "",\n            "Rejected paths / do not revisit unless necessary:",\n            rejected_lines,\n            "",\n            "Open questions:",\n            open_question_lines,\n            "",\n            "Where we left off:",\n            next_step_lines,\n            "",\n            "Continue from this exact state.",\n            "My next request:",\n            "- ",\n        ]\n    )\n\n\ndef sanitize_model_brief(text: Any) -> str:\n    value = str(text or "")\n    value = re.sub(r"^```[a-z]*\\n?", "", value, flags=re.IGNORECASE)\n    value = re.sub(r"\\n```$", "", value, flags=re.IGNORECASE)\n    return value.strip()\n\n\ndef extract_tagged_rationale(text: str) -> tuple[str, str]:\n    chunks: list[str] = []\n\n    def _replace(match: re.Match[str]) -> str:\n        cleaned = sanitize_model_brief(match.group(1))\n        if cleaned:\n            chunks.append(cleaned)\n        return ""\n\n    remaining = THINK_BLOCK_PATTERN.sub(_replace, text)\n    return "\\n\\n".join(chunks).strip(), remaining.strip()\n\n\ndef split_model_output(raw_text: str) -> tuple[str, str]:\n    sanitized_raw = sanitize_model_brief(raw_text).replace("\\r", "")\n    auxiliary_rationale, remaining = extract_tagged_rationale(sanitized_raw)\n    return remaining or sanitized_raw, auxiliary_rationale\n\n\ndef normalize_generated_brief(text: str) -> str:\n    brief = sanitize_model_brief(text).replace("\\r", "")\n    brief = re.sub(r"\\n{3,}", "\\n\\n", brief)\n    brief = re.sub(r"(?m)^-\\s*", "- ", brief)\n    brief = re.sub(\n        rf"^(?:{re.escape(PREAMBLE)}\\s*\\n+){{2,}}",\n        f"{PREAMBLE}\\n\\n",\n        brief,\n        count=1,\n    )\n    for section, pattern in INLINE_SECTION_PATTERNS.items():\n        brief = pattern.sub(lambda match, section=section: f"{section}\\n- {match.group(1).strip()}", brief)\n    for pattern, replacement in SECTION_HEADER_NORMALIZATIONS:\n        brief = pattern.sub(replacement, brief)\n    brief = re.sub(r"(?m)^My next request:\\s*$\\n(?!-)", "My next request:\\n-", brief)\n    brief = re.sub(r"(?m)^My next request:\\n-\\s*$", "My next request:\\n-", brief)\n    return brief.strip()\n\n\ndef has_required_sections(text: str) -> bool:\n    return all(section in text for section in REQUIRED_SECTIONS)\n\n\ndef has_duplicate_or_missing_section_lines(text: str) -> bool:\n    return any(len(pattern.findall(text)) != 1 for pattern in SECTION_LINE_PATTERNS.values())\n\n\ndef has_prompt_echo(text: str) -> bool:\n    return any(marker in text[:1200] for marker in PROMPT_ECHO_MARKERS)\n\n\ndef has_single_next_request_bullet(text: str) -> bool:\n    match = re.search(r"(?ms)^My next request:\\n(.*)\\Z", text)\n    if not match:\n        return False\n    lines = [line.strip() for line in match.group(1).splitlines() if line.strip()]\n    return len(lines) == 1 and (lines[0] == "-" or lines[0].startswith("- "))\n\n\ndef describe_invalid_brief(text: str) -> list[str]:\n    issues: list[str] = []\n    missing_sections = [section for section in REQUIRED_SECTIONS if section not in text]\n    if missing_sections:\n        preview = ", ".join(missing_sections[:4])\n        if len(missing_sections) > 4:\n            preview = f"{preview}, +{len(missing_sections) - 4} more"\n        issues.append(f"missing sections: {preview}")\n\n    for section, pattern in SECTION_LINE_PATTERNS.items():\n        count = len(pattern.findall(text))\n        if count == 0:\n            issues.append(f"missing section line: {section}")\n        elif count > 1:\n            issues.append(f"duplicate section line: {section} x{count}")\n\n    if not has_single_next_request_bullet(text):\n        issues.append("invalid My next request bullet")\n    if PLACEHOLDER_BULLET_PATTERN.search(text):\n        issues.append("contains placeholder bullet")\n    if has_prompt_echo(text):\n        issues.append("prompt echo in opening text")\n    if not issues:\n        issues.append("unknown validation failure")\n    return issues\n\n\ndef summarize_issues(issues: list[str], limit: int = 4) -> str:\n    if not issues:\n        return "unknown validation failure"\n    summary = "; ".join(issues[:limit])\n    if len(issues) > limit:\n        summary = f"{summary}; +{len(issues) - limit} more"\n    return summary\n\n\ndef is_valid_generated_brief(text: str) -> bool:\n    return (\n        has_required_sections(text)\n        and not has_duplicate_or_missing_section_lines(text)\n        and has_single_next_request_bullet(text)\n        and not PLACEHOLDER_BULLET_PATTERN.search(text)\n        and not has_prompt_echo(text)\n    )\n\n\ndef rewrite_tags(tags: list[str], model: str, has_rationale: bool) -> list[str]:\n    filtered = [\n        tag\n        for tag in list(tags or [])\n        if tag\n        not in {\n            "teacher-draft-backfilled",\n            "teacher-draft-saved",\n            "teacher-draft-generated",\n            "teacher-draft-failed",\n            "teacher-rationale-captured",\n            "local-heuristic",\n        }\n        and not str(tag).startswith("local-model:")\n    ]\n    filtered.extend(["teacher-draft-generated", f"local-model:{model}"])\n    if has_rationale:\n        filtered.append("teacher-rationale-captured")\n    unique: list[str] = []\n    for tag in filtered:\n        if tag not in unique:\n            unique.append(tag)\n    return unique\n\n\ndef rewrite_failure_tags(tags: list[str], model: str) -> list[str]:\n    filtered = [\n        tag\n        for tag in list(tags or [])\n        if tag\n        not in {\n            "teacher-draft-backfilled",\n            "teacher-draft-saved",\n            "teacher-draft-generated",\n            "teacher-draft-failed",\n            "teacher-rationale-captured",\n            "local-heuristic",\n        }\n        and not str(tag).startswith("local-model:")\n    ]\n    filtered.extend(["teacher-draft-failed", f"local-model:{model}"])\n    unique: list[str] = []\n    for tag in filtered:\n        if tag not in unique:\n            unique.append(tag)\n    return unique\n\n\ndef finalize_failed_row(row: dict[str, Any], model: str) -> dict[str, Any]:\n    next_row = dict(row)\n    next_row.pop("teacher_draft_brief", None)\n    next_row.pop("auxiliary_rationale", None)\n    next_row.pop("auxiliary_rationale_format", None)\n    next_row["tags"] = rewrite_failure_tags(list(row.get("tags", [])), model)\n    return next_row\n\n\ndef count_rows_with_tag(rows: list[dict[str, Any]], tag: str) -> int:\n    return sum(1 for row in rows if tag in list(row.get("tags", [])))\n\n\ndef build_generation_attempts(requested_max_new_tokens: int) -> list[dict[str, int]]:\n    raw_budgets = [\n        DEFAULT_TRANSCRIPT_CHAR_BUDGET,\n        min(DEFAULT_TRANSCRIPT_CHAR_BUDGET, 10000),\n        min(DEFAULT_TRANSCRIPT_CHAR_BUDGET, 7000),\n        min(DEFAULT_TRANSCRIPT_CHAR_BUDGET, 5000),\n    ]\n    raw_token_caps = [\n        requested_max_new_tokens,\n        min(requested_max_new_tokens, 320),\n        min(requested_max_new_tokens, 256),\n        min(requested_max_new_tokens, 192),\n    ]\n    attempts: list[dict[str, int]] = []\n    for transcript_char_budget, max_new_tokens in zip(raw_budgets, raw_token_caps):\n        attempt = {\n            "transcript_char_budget": transcript_char_budget,\n            "max_new_tokens": max_new_tokens,\n        }\n        if attempt not in attempts:\n            attempts.append(attempt)\n    return attempts\n\n\ndef is_cuda_oom_error(exc: BaseException) -> bool:\n    message = str(exc).lower()\n    return "out of memory" in message and ("cuda" in message or "cublas" in message)\n\n\ndef clear_torch_memory(device: torch.device | None) -> None:\n    gc.collect()\n    if not device or device.type != "cuda" or not torch.cuda.is_available():\n        return\n    try:\n        torch.cuda.empty_cache()\n    except RuntimeError:\n        pass\n    try:\n        torch.cuda.ipc_collect()\n    except RuntimeError:\n        pass\n\n\ndef detect_single_device() -> torch.device:\n    if not torch.cuda.is_available():\n        raise RuntimeError("This notebook requires a CUDA GPU.")\n    return torch.device("cuda")\n\n\ndef resolve_model_load_settings(device_index: int = 0) -> dict[str, Any]:\n    major, minor = torch.cuda.get_device_capability(device_index)\n    capability = (major, minor)\n    if capability < (7, 0):\n        return {\n            "capability": capability,\n            "torch_dtype": torch.float16,\n            "attn_implementation": "eager",\n            "compatibility_mode": "legacy_cuda",\n        }\n    return {\n        "capability": capability,\n        "torch_dtype": torch.float16,\n        "attn_implementation": "sdpa",\n        "compatibility_mode": "default",\n    }\n\n\ndef list_input_json_candidates(root: str = "/kaggle/input") -> list[str]:\n    input_root = Path(root)\n    if not input_root.exists():\n        return []\n    return [str(path) for path in sorted(input_root.rglob("*.json"))]\n\n\ndef resolve_explicit_input_path(explicit_input_path: str) -> str:\n    candidate = Path(explicit_input_path)\n    if candidate.is_file():\n        return str(candidate)\n    if candidate.is_dir():\n        json_candidates = sorted(candidate.rglob("*.json"))\n        if len(json_candidates) == 1:\n            return str(json_candidates[0])\n        for preferred_name in (\n            "teacher_input_rows.json",\n            "live_extension_review_queue_parallel_assistant_repaired_v11.json",\n        ):\n            for json_candidate in json_candidates:\n                if json_candidate.name == preferred_name:\n                    return str(json_candidate)\n        preview = "\\n".join(str(path) for path in json_candidates) if json_candidates else "(no JSON files found)"\n        raise FileNotFoundError(\n            f"Explicit input directory does not resolve to a single JSON file: {candidate}\\n"\n            f"Candidates:\\n{preview}"\n        )\n    raise FileNotFoundError(f"Explicit input path does not exist: {candidate}")\n\n\ndef resolve_known_input_path(run_dataset: str) -> str:\n    dataset_key = run_dataset.strip().lower()\n    if dataset_key not in KNOWN_INPUT_PATHS:\n        raise ValueError(f"Unknown RUN_DATASET value: {run_dataset}")\n    candidate = Path(KNOWN_INPUT_PATHS[dataset_key])\n    if candidate.exists():\n        return str(candidate)\n    available = list_input_json_candidates()\n    preview = "\\n".join(available) if available else "(no JSON files found under /kaggle/input)"\n    raise FileNotFoundError(\n        f"Expected input JSON is missing for RUN_DATASET={run_dataset}: {candidate}\\n"\n        f"Available candidates:\\n{preview}"\n    )\n\n\ndef resolve_input_path(run_dataset: str, explicit_input_path: str = "") -> str:\n    explicit_value = explicit_input_path.strip()\n    if explicit_value:\n        return resolve_explicit_input_path(explicit_value)\n    return resolve_known_input_path(run_dataset)\n\n\ndef load_json_rows(path: str) -> list[dict[str, Any]]:\n    data = json.loads(Path(path).read_text(encoding="utf-8"))\n    if not isinstance(data, list):\n        raise RuntimeError(f"{path} does not contain a JSON array.")\n    return data\n\n\ndef select_rows(rows: list[dict[str, Any]], limit: int = 0) -> list[dict[str, Any]]:\n    return rows[:limit] if limit > 0 else list(rows)\n\n\ndef derive_failure_debug_path(workdir: str, run_mode: str) -> str:\n    return str(Path(workdir) / f"{run_mode}.failures.ndjson")\n\n\ndef append_failure_debug_record(debug_path: str, payload: dict[str, Any]) -> None:\n    ensure_parent(debug_path)\n    with Path(debug_path).open("a", encoding="utf-8") as handle:\n        handle.write(json.dumps(payload, ensure_ascii=False) + "\\n")\n\n\ndef reset_run_artifacts(*paths: str) -> None:\n    for path in paths:\n        if path and Path(path).exists():\n            Path(path).unlink()\n\n\ndef load_resume_rows(output_path: str, expected_rows: int) -> list[dict[str, Any]]:\n    path = Path(output_path)\n    if not path.exists():\n        return []\n    loaded = json.loads(path.read_text(encoding="utf-8"))\n    if not isinstance(loaded, list):\n        raise RuntimeError("Existing output file does not contain a JSON array.")\n    if len(loaded) > expected_rows:\n        raise RuntimeError("Existing output file has more rows than the selected run.")\n    return loaded\n\n\ndef validate_resume_summary(\n    summary_output_path: str | None,\n    *,\n    input_path: str,\n    model: str,\n    run_mode: str,\n    requested_rows: int,\n) -> None:\n    if not summary_output_path:\n        return\n    summary_path = Path(summary_output_path)\n    if not summary_path.exists():\n        return\n    summary = json.loads(summary_path.read_text(encoding="utf-8"))\n    checks = {\n        "inputPath": input_path,\n        "model": model,\n        "runMode": run_mode,\n        "requestedRows": requested_rows,\n    }\n    mismatches = [\n        f"{key}={summary.get(key)!r} expected {value!r}"\n        for key, value in checks.items()\n        if summary.get(key) != value\n    ]\n    if mismatches:\n        raise RuntimeError("Resume refused because prior summary does not match this run: " + "; ".join(mismatches))\n\n\ndef build_summary(\n    *,\n    model: str,\n    input_path: str,\n    run_mode: str,\n    requested_rows: int,\n    completed_rows: int,\n    generated_rows: list[dict[str, Any]],\n    failures: list[dict[str, str]],\n    capture_rationale: bool,\n    started_at: float,\n    resumed_rows: int,\n    status: str,\n    failure_debug_path: str,\n) -> dict[str, Any]:\n    success_count = count_rows_with_tag(generated_rows, "teacher-draft-generated")\n    failure_count = count_rows_with_tag(generated_rows, "teacher-draft-failed")\n    shard_complete = completed_rows == requested_rows\n    return {\n        "generatedAt": time.strftime("%Y-%m-%dT%H:%M:%S%z"),\n        "durationMs": round((time.time() - started_at) * 1000),\n        "model": model,\n        "inputPath": input_path,\n        "runMode": run_mode,\n        "inputRows": requested_rows,\n        "requestedRows": requested_rows,\n        "completedRows": completed_rows,\n        "mergedRows": completed_rows,\n        "generatedRows": success_count,\n        "captureRationale": capture_rationale,\n        "rationaleCapturedRows": sum(1 for row in generated_rows if str(row.get("auxiliary_rationale", "")).strip()),\n        "failedRows": failure_count,\n        "failures": failures,\n        "numShards": 1,\n        "completeShards": 1 if shard_complete else 0,\n        "shards": [\n            {\n                "shardIndex": 0,\n                "path": "single-run",\n                "expectedRows": requested_rows,\n                "mergedRows": completed_rows,\n                "complete": shard_complete,\n            }\n        ],\n        "shardIndex": 0,\n        "resumedRows": resumed_rows,\n        "status": status,\n        "failureDebugPath": failure_debug_path,\n    }\n\n\ndef write_checkpoint(\n    *,\n    output_path: str,\n    summary_output: str | None,\n    generated_rows: list[dict[str, Any]],\n    summary: dict[str, Any],\n) -> None:\n    ensure_parent(output_path)\n    Path(output_path).write_text(json.dumps(generated_rows, indent=2) + "\\n", encoding="utf-8")\n    if summary_output:\n        ensure_parent(summary_output)\n        Path(summary_output).write_text(json.dumps(summary, indent=2) + "\\n", encoding="utf-8")\n\n\ndef read_failure_records(debug_path: str, limit: int = 5) -> list[dict[str, Any]]:\n    path = Path(debug_path)\n    if not path.exists():\n        return []\n    records: list[dict[str, Any]] = []\n    with path.open("r", encoding="utf-8") as handle:\n        for line in handle:\n            if len(records) >= limit:\n                break\n            line = line.strip()\n            if not line:\n                continue\n            records.append(json.loads(line))\n    return records\n\n\nclass HFTeacherGenerator:\n    def __init__(self, model_id: str, temperature: float, max_new_tokens: int) -> None:\n        self.model_id = model_id\n        self.device = detect_single_device()\n        self.model_load_settings = resolve_model_load_settings(0)\n        self.cuda_capability = self.model_load_settings["capability"]\n        self.attn_implementation = self.model_load_settings["attn_implementation"]\n        self.torch_dtype = self.model_load_settings["torch_dtype"]\n        self.compatibility_mode = self.model_load_settings["compatibility_mode"]\n        self.temperature = temperature\n        self.max_new_tokens = max_new_tokens\n        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=False)\n        if self.tokenizer.pad_token is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        model_kwargs: dict[str, Any] = {\n            "trust_remote_code": True,\n            "torch_dtype": self.torch_dtype,\n            "attn_implementation": self.attn_implementation,\n        }\n        try:\n            self.model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)\n        except TypeError:\n            model_kwargs.pop("attn_implementation", None)\n            self.model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)\n        self.model.to(self.device)\n        self.model.eval()\n\n    def generate(self, prompt: str, capture_rationale: bool, *, max_new_tokens: int | None = None) -> str:\n        messages = [{"role": "user", "content": prompt}]\n        kwargs: dict[str, Any] = {\n            "add_generation_prompt": True,\n            "tokenize": True,\n            "return_dict": True,\n            "return_tensors": "pt",\n        }\n        if hasattr(self.tokenizer, "apply_chat_template") and self.tokenizer.chat_template:\n            if "qwen" in self.model_id.lower() and not capture_rationale:\n                kwargs["enable_thinking"] = False\n            try:\n                inputs = self.tokenizer.apply_chat_template(messages, **kwargs)\n            except TypeError:\n                kwargs.pop("enable_thinking", None)\n                inputs = self.tokenizer.apply_chat_template(messages, **kwargs)\n        else:\n            inputs = self.tokenizer(prompt, return_tensors="pt")\n\n        inputs = inputs.to(self.device)\n        generated = None\n        new_tokens = None\n        try:\n            with torch.inference_mode():\n                generated = self.model.generate(\n                    **inputs,\n                    do_sample=self.temperature > 0,\n                    temperature=max(self.temperature, 1e-5),\n                    max_new_tokens=max_new_tokens or self.max_new_tokens,\n                    pad_token_id=self.tokenizer.pad_token_id,\n                    eos_token_id=self.tokenizer.eos_token_id,\n                )\n            new_tokens = generated[:, inputs["input_ids"].shape[1] :]\n            return self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]\n        finally:\n            del inputs\n            if new_tokens is not None:\n                del new_tokens\n            if generated is not None:\n                del generated\n            clear_torch_memory(self.device)\n\n\ndef generate_teacher_draft(\n    *,\n    generator: HFTeacherGenerator,\n    row: dict[str, Any],\n    capture_rationale: bool,\n) -> tuple[str, str]:\n    attempts = build_generation_attempts(generator.max_new_tokens)\n    last_oom_error: RuntimeError | None = None\n    attempt_debug_records: list[dict[str, Any]] = []\n\n    for attempt_index, attempt in enumerate(attempts, start=1):\n        prompt = build_prompt(\n            row,\n            capture_rationale,\n            transcript_char_budget=attempt["transcript_char_budget"],\n        )\n        try:\n            raw_output = generator.generate(\n                prompt,\n                capture_rationale,\n                max_new_tokens=attempt["max_new_tokens"],\n            )\n            brief_text, auxiliary_rationale = split_model_output(raw_output)\n            brief = normalize_generated_brief(brief_text)\n            initial_brief = brief\n            initial_issues = describe_invalid_brief(brief) if not is_valid_generated_brief(brief) else []\n\n            if initial_issues:\n                print(\n                    f"Invalid brief for \'{row.get(\'title\', \'\')}\' on attempt {attempt_index}/{len(attempts)}: "\n                    f"{summarize_issues(initial_issues)}",\n                    flush=True,\n                )\n                retry_prompt = (\n                    f"{prompt}\\n\\nThe previous answer was invalid because it repeated the instructions, "\n                    "used placeholders, or missed the required structure. Regenerate and output only the "\n                    "final continuation brief with filled sections and no analysis."\n                )\n                retry_raw_output = generator.generate(\n                    retry_prompt,\n                    capture_rationale,\n                    max_new_tokens=attempt["max_new_tokens"],\n                )\n                brief_text, auxiliary_rationale = split_model_output(retry_raw_output)\n                brief = normalize_generated_brief(brief_text)\n                retry_issues = describe_invalid_brief(brief) if not is_valid_generated_brief(brief) else []\n\n                if retry_issues:\n                    print(\n                        f"Retry still invalid for \'{row.get(\'title\', \'\')}\' on attempt {attempt_index}/{len(attempts)}: "\n                        f"{summarize_issues(retry_issues)}",\n                        flush=True,\n                    )\n                    attempt_debug_records.append(\n                        {\n                            "attemptIndex": attempt_index,\n                            "transcriptCharBudget": attempt["transcript_char_budget"],\n                            "maxNewTokens": attempt["max_new_tokens"],\n                            "promptLength": len(prompt),\n                            "promptPreview": trim_debug_text(prompt),\n                            "initialRawOutput": trim_debug_text(raw_output),\n                            "initialNormalizedBrief": trim_debug_text(initial_brief),\n                            "initialIssues": initial_issues,\n                            "retryRawOutput": trim_debug_text(retry_raw_output),\n                            "retryNormalizedBrief": trim_debug_text(brief),\n                            "retryIssues": retry_issues,\n                        }\n                    )\n                    continue\n\n            return brief, auxiliary_rationale\n        except RuntimeError as exc:\n            if not is_cuda_oom_error(exc):\n                raise\n            last_oom_error = exc\n            print(\n                "OOM while generating "\n                f"\'{row.get(\'title\', \'\')}\' on attempt {attempt_index}/{len(attempts)}; "\n                f"retrying with transcript_char_budget={attempt[\'transcript_char_budget\']} "\n                f"and max_new_tokens={attempt[\'max_new_tokens\']}",\n                flush=True,\n            )\n            clear_torch_memory(generator.device)\n\n    if attempt_debug_records:\n        issues: list[str] = []\n        for record in attempt_debug_records:\n            for issue in record.get("retryIssues") or record.get("initialIssues") or []:\n                if issue not in issues:\n                    issues.append(issue)\n        raise MalformedBriefError(issues, {"attempts": attempt_debug_records})\n\n    if last_oom_error is not None:\n        raise RuntimeError(f"{last_oom_error} after {len(attempts)} memory-recovery attempts.") from last_oom_error\n\n    raise RuntimeError("Teacher draft generation exhausted all attempts without producing output.")\n\n\ndef run_generation(\n    *,\n    generator: HFTeacherGenerator,\n    rows: list[dict[str, Any]],\n    input_path: str,\n    output_path: str,\n    summary_output: str | None,\n    workdir: str,\n    model: str,\n    run_mode: str,\n    checkpoint_every: int,\n    capture_rationale: bool,\n    resume: bool,\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    started_at = time.time()\n    selected_rows = list(rows)\n    failure_debug_path = derive_failure_debug_path(workdir, run_mode)\n\n    if resume:\n        validate_resume_summary(\n            summary_output,\n            input_path=input_path,\n            model=model,\n            run_mode=run_mode,\n            requested_rows=len(selected_rows),\n        )\n        generated_rows = load_resume_rows(output_path, len(selected_rows))\n        if generated_rows:\n            print(f"Resuming {run_mode} run from row {len(generated_rows) + 1}/{len(selected_rows)}", flush=True)\n    else:\n        reset_run_artifacts(output_path, summary_output or "", failure_debug_path)\n        generated_rows = []\n\n    failures: list[dict[str, str]] = []\n    resumed_rows = len(generated_rows)\n\n    for index, row in enumerate(selected_rows[resumed_rows:], start=resumed_rows + 1):\n        print(f"Generating {index}/{len(selected_rows)}: {row.get(\'title\', \'\')}", flush=True)\n        try:\n            brief, auxiliary_rationale = generate_teacher_draft(\n                generator=generator,\n                row=row,\n                capture_rationale=capture_rationale,\n            )\n            next_row = dict(row)\n            next_row["teacher_draft_brief"] = brief\n            if auxiliary_rationale:\n                next_row["auxiliary_rationale"] = auxiliary_rationale\n                next_row["auxiliary_rationale_format"] = "visible_cot"\n            next_row["tags"] = rewrite_tags(list(row.get("tags", [])), model, bool(auxiliary_rationale))\n            generated_rows.append(next_row)\n        except Exception as exc:  # noqa: BLE001\n            failure_record = {\n                "title": str(row.get("title", "")),\n                "conversation_id": str(row.get("conversation_id", "")),\n                "message": str(exc),\n            }\n            debug_payload: dict[str, Any] = {\n                "title": failure_record["title"],\n                "conversation_id": failure_record["conversation_id"],\n                "provider": str(row.get("provider", "")),\n                "model": model,\n                "runMode": run_mode,\n                "errorType": type(exc).__name__,\n                "message": str(exc),\n            }\n            if isinstance(exc, MalformedBriefError):\n                debug_payload.update(exc.debug_payload)\n            append_failure_debug_record(failure_debug_path, debug_payload)\n            print(\n                f"Logged failure debug for \'{failure_record[\'title\']}\' to {failure_debug_path}",\n                flush=True,\n            )\n            failures.append(failure_record)\n            generated_rows.append(finalize_failed_row(row, model))\n\n        if len(generated_rows) % max(checkpoint_every, 1) == 0 or index == len(selected_rows):\n            summary = build_summary(\n                model=model,\n                input_path=input_path,\n                run_mode=run_mode,\n                requested_rows=len(selected_rows),\n                completed_rows=len(generated_rows),\n                generated_rows=generated_rows,\n                failures=failures,\n                capture_rationale=capture_rationale,\n                started_at=started_at,\n                resumed_rows=resumed_rows,\n                status="running" if index < len(selected_rows) else "completed",\n                failure_debug_path=failure_debug_path,\n            )\n            write_checkpoint(\n                output_path=output_path,\n                summary_output=summary_output,\n                generated_rows=generated_rows,\n                summary=summary,\n            )\n\n    final_summary = build_summary(\n        model=model,\n        input_path=input_path,\n        run_mode=run_mode,\n        requested_rows=len(selected_rows),\n        completed_rows=len(generated_rows),\n        generated_rows=generated_rows,\n        failures=failures,\n        capture_rationale=capture_rationale,\n        started_at=started_at,\n        resumed_rows=resumed_rows,\n        status="completed",\n        failure_debug_path=failure_debug_path,\n    )\n    write_checkpoint(\n        output_path=output_path,\n        summary_output=summary_output,\n        generated_rows=generated_rows,\n        summary=final_summary,\n    )\n    return generated_rows, final_summary\n'
RUNTIME_PATH = Path("/kaggle/working/teacher_runtime.py")
RUNTIME_PATH.write_text(EMBEDDED_RUNTIME_SOURCE, encoding="utf-8")

if "teacher_runtime" in sys.modules:
    del sys.modules["teacher_runtime"]

import torch
spec = importlib.util.spec_from_file_location("teacher_runtime", RUNTIME_PATH)
teacher_runtime = importlib.util.module_from_spec(spec)
sys.modules["teacher_runtime"] = teacher_runtime
spec.loader.exec_module(teacher_runtime)

DEFAULT_MODEL = teacher_runtime.DEFAULT_MODEL
HFTeacherGenerator = teacher_runtime.HFTeacherGenerator
KNOWN_INPUT_PATHS = teacher_runtime.KNOWN_INPUT_PATHS
list_input_json_candidates = teacher_runtime.list_input_json_candidates
load_json_rows = teacher_runtime.load_json_rows
read_failure_records = teacher_runtime.read_failure_records
resolve_input_path = teacher_runtime.resolve_input_path
run_generation = teacher_runtime.run_generation
select_rows = teacher_runtime.select_rows

print(f"Embedded runtime written to: {RUNTIME_PATH}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Torch CUDA build: {torch.version.cuda}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability(0)
    print(f"Primary GPU: {torch.cuda.get_device_name(0)}")
    print(f"Primary GPU capability: {capability[0]}.{capability[1]}")


In [ ]:
RUN_DATASET = "full"  # full | repair
INPUT_PATH_OVERRIDE = "/kaggle/input/datasets/sujendragharat/qwen-4b-teacher-2gpu-inputs-new"
MODEL_ID = DEFAULT_MODEL
SMOKE_LIMIT = 10
RESUME = False
CHECKPOINT_EVERY = 1
CAPTURE_RATIONALE = False
MAX_NEW_TOKENS = 420
TEMPERATURE = 0.2
ALLOW_FULL_RUN = False

NOTEBOOK_WORKDIR = "/kaggle/working/qwen_teacher_t4_notebook"
SMOKE_OUTPUT_PATH = f"{NOTEBOOK_WORKDIR}/smoke/qwen4b_teacher_drafts_merged.json"
SMOKE_SUMMARY_OUTPUT = f"{NOTEBOOK_WORKDIR}/smoke/qwen4b_teacher_drafts_merged.summary.json"
FULL_OUTPUT_PATH = "/kaggle/working/qwen4b_teacher_drafts_merged.json"
FULL_SUMMARY_OUTPUT = "/kaggle/working/qwen4b_teacher_drafts_merged.summary.json"

print(json.dumps({
    "RUN_DATASET": RUN_DATASET,
    "INPUT_PATH_OVERRIDE": INPUT_PATH_OVERRIDE,
    "MODEL_ID": MODEL_ID,
    "SMOKE_LIMIT": SMOKE_LIMIT,
    "RESUME": RESUME,
    "CHECKPOINT_EVERY": CHECKPOINT_EVERY,
    "CAPTURE_RATIONALE": CAPTURE_RATIONALE,
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "TEMPERATURE": TEMPERATURE,
    "ALLOW_FULL_RUN": ALLOW_FULL_RUN,
    "NOTEBOOK_WORKDIR": NOTEBOOK_WORKDIR,
    "SMOKE_OUTPUT_PATH": SMOKE_OUTPUT_PATH,
    "SMOKE_SUMMARY_OUTPUT": SMOKE_SUMMARY_OUTPUT,
    "FULL_OUTPUT_PATH": FULL_OUTPUT_PATH,
    "FULL_SUMMARY_OUTPUT": FULL_SUMMARY_OUTPUT,
}, indent=2))


In [ ]:
INPUT_PATH = resolve_input_path(RUN_DATASET, INPUT_PATH_OVERRIDE)
ALL_ROWS = load_json_rows(INPUT_PATH)

print(f"Resolved INPUT_PATH: {INPUT_PATH}")
print("Known mounted paths:")
print(json.dumps(KNOWN_INPUT_PATHS, indent=2))
print(f"Rows loaded: {len(ALL_ROWS)}")
print("Sample titles:")
for row in ALL_ROWS[:5]:
    print("-", row.get("title", ""))

print("Visible JSON candidates under /kaggle/input:")
for path in list_input_json_candidates():
    print("-", path)


In [ ]:
GENERATOR = HFTeacherGenerator(MODEL_ID, TEMPERATURE, MAX_NEW_TOKENS)
print(f"Loaded model: {GENERATOR.model_id}")
print(f"Effective device: {GENERATOR.device}")
print(f"CUDA capability: {GENERATOR.cuda_capability[0]}.{GENERATOR.cuda_capability[1]}")
print(f"Compatibility mode: {GENERATOR.compatibility_mode}")
print(f"Attention implementation: {GENERATOR.attn_implementation}")
print(f"Requested torch dtype: {GENERATOR.torch_dtype}")
print(f"Model dtype: {GENERATOR.model.dtype}")


In [ ]:
SMOKE_ROWS = select_rows(ALL_ROWS, SMOKE_LIMIT)
SMOKE_GENERATED_ROWS, SMOKE_SUMMARY = run_generation(
    generator=GENERATOR,
    rows=SMOKE_ROWS,
    input_path=INPUT_PATH,
    output_path=SMOKE_OUTPUT_PATH,
    summary_output=SMOKE_SUMMARY_OUTPUT,
    workdir=NOTEBOOK_WORKDIR,
    model=MODEL_ID,
    run_mode="smoke",
    checkpoint_every=CHECKPOINT_EVERY,
    capture_rationale=CAPTURE_RATIONALE,
    resume=False,
)
print(json.dumps(SMOKE_SUMMARY, indent=2))


In [ ]:
SMOKE_FAILURES = read_failure_records(SMOKE_SUMMARY["failureDebugPath"], limit=5)
SMOKE_PASS = SMOKE_SUMMARY["generatedRows"] >= 8 and SMOKE_SUMMARY["failedRows"] <= 2

print(f"Smoke pass: {SMOKE_PASS}")
print(json.dumps(SMOKE_SUMMARY, indent=2))
print("Sample smoke failure records:")
for record in SMOKE_FAILURES:
    print(json.dumps(record, indent=2)[:4000])
    print("-" * 80)


In [ ]:
if not ALLOW_FULL_RUN:
    raise RuntimeError("Set ALLOW_FULL_RUN = True in the config cell after reviewing the smoke run.")
if "SMOKE_PASS" not in globals() or not SMOKE_PASS:
    raise RuntimeError("Smoke run did not pass. Review the smoke outputs before starting the full run.")

FULL_GENERATED_ROWS, FULL_SUMMARY = run_generation(
    generator=GENERATOR,
    rows=ALL_ROWS,
    input_path=INPUT_PATH,
    output_path=FULL_OUTPUT_PATH,
    summary_output=FULL_SUMMARY_OUTPUT,
    workdir=NOTEBOOK_WORKDIR,
    model=MODEL_ID,
    run_mode="full",
    checkpoint_every=CHECKPOINT_EVERY,
    capture_rationale=CAPTURE_RATIONALE,
    resume=RESUME,
)
print(json.dumps(FULL_SUMMARY, indent=2))


In [ ]:
print(json.dumps(FULL_SUMMARY, indent=2))

passing_rows = [row for row in FULL_GENERATED_ROWS if "teacher-draft-generated" in row.get("tags", [])]
print("Sample passing rows:")
for row in passing_rows[:3]:
    print("=" * 80)
    print(row.get("title", ""))
    print(row.get("teacher_draft_brief", "")[:2000])

print("Sample final failure records:")
for record in read_failure_records(FULL_SUMMARY["failureDebugPath"], limit=5):
    print(json.dumps(record, indent=2)[:4000])
    print("-" * 80)
